# TerraX in Jupyter

TerraX's tools on local files, with the same code and the same numbers as the web app, and their outputs on an interactive map. Install with `pip install -e 'backend[notebook]'` from the TerraX repository. Each cell below uses the bundled synthetic samples; point the paths at your own GeoTIFFs.

In [ ]:
from pathlib import Path

from terrax.local import run, tools
from terrax.notebook import show

# The bundled samples (this notebook lives in notebooks/, next to backend/).
SAMPLES = Path.cwd().parent / "backend" / "data" / "samples"
if not SAMPLES.exists():
    SAMPLES = Path.cwd() / "backend" / "data" / "samples"
sorted(tools())

## Forest loss between two dates

NDVI change: forest at the start (NDVI ≥ 0.5), loss where NDVI drops by 0.2 or more. The run returns the numbers, the report and the output files.

In [ ]:
forest = run("forest", {"a": SAMPLES / "forest_ndvi_2016_synthetic.tif", "b": SAMPLES / "forest_ndvi_2024_synthetic.tif"},
             {"mode": "ndvi", "mmuHa": 0.5}, out="out/forest")
loss = forest["result"]["loss"]["ha"]
print(f"Forest loss: {loss:,.2f} ha; files: {[f.name for f in forest.files]}")

In [ ]:
forest  # a result shows as its report

In [ ]:
show(forest)  # the class map and loss polygons, with a legend

## Hydrology: streams, flow accumulation and wetness

D∞ flow (Tarboton 1997) adds the topographic wetness index; flow accumulation is drawn on a log scale.

In [ ]:
hydro = run("hydrology", {"dem": SAMPLES / "terrain_dem_synthetic.tif"}, {"method": "dinf", "thresholdKm2": 0.5}, out="out/hydrology")
print(hydro["wetness"]["summary"])
show(hydro)

## Canopy height and tree tops from LiDAR

In [ ]:
canopy = run("canopy", {"cloud": SAMPLES / "lidar_forest_synthetic.laz"}, {"cell": 1, "minTree": 5}, out="out/canopy")
print(canopy["canopy"]["trees"]["count"], "tree tops")
show(canopy)

## Disturbances dated per pixel in an image series

In [ ]:
series = run("stack", {"files": sorted(SAMPLES.glob("series_ndvi_*_synthetic.tif"))}, {"index": "ndvi", "mapBreaks": True, "minDrop": 0.15}, out="out/series")
for x in series["breaks"]["byDate"]:
    print(x["date"], round(x["ha"], 2), "ha")
show(series, layers=["disturbance_date.tif"])

The outputs are ordinary GeoTIFF and GeoJSON files: open them with rasterio, geopandas or xarray for further work, or load them in QGIS.

In [ ]:
import rasterio

with rasterio.open("out/forest/forest_ndvi_2016_synthetic_forest_change.tif") as d:
    classes = d.read(1)
    print(d.crs, d.res, {int(v): int((classes == v).sum()) for v in (1, 2, 3, 4)})